# 04 — Mapas de peligro acústico
*El mapa de peligro, honesto.*

Danger score v2 (ponderado por severidad y, si existe, por precisión de NB-02; con incertidumbre por bootstrap), mapa coroplético por celda, mapa interactivo Folium y trayectoria coloreada por velocidad.

> Solo **Horn + Siren** suman al danger score. Speech es contexto.

In [ ]:
import sys, warnings
sys.path.insert(0, "../scripts")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geo_utils as gu

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.width", 140)

geo = gu.load_geo()
geo = geo[~geo["trayecto"].isin(gu.BAD_TRAYECTOS)].copy()
geo["class_name"] = geo["class"].map(gu.CLASS_NAMES)
print(f"{len(geo):,} detecciones geolocalizadas | fuentes: {geo['source'].unique().tolist()}")
geo.head(3)

In [ ]:
import folium
from folium.plugins import HeatMap
from pathlib import Path

# Pesos de precisión por clase (de NB-02 si la validación está hecha; si no, 1.0)
prec_path = Path("../validation/precision_by_class.csv")
if prec_path.exists():
    pw = pd.read_csv(prec_path).set_index("class")["precision"].to_dict()
    print("Pesos de precisión cargados:", pw)
else:
    pw = {c: 1.0 for c in gu.SEVERITY}
    print("Sin validación: precisión=1.0 para todas las clases (declarar en memoria).")

road = geo[geo["class"].isin(gu.ROAD_CLASSES)].copy()
road["w"] = road.apply(lambda r: gu.SEVERITY[r["class"]] * pw.get(r["class"], 1.0), axis=1)
print(f"{len(road)} eventos viales (Horn={int((road['class']==0).sum())}, Siren={int((road['class']==1).sum())})")

## C1 — Danger score v2 en grilla + incertidumbre (bootstrap)
Score por celda = Σ severidad×precisión, normalizado 0–100. El IC por bootstrap (remuestreo de trayectos) cuantifica cuánto depende cada celda de un trayecto concreto.

In [ ]:
CELL = 0.0005  # ~50 m
def cell_scores(df):
    gi = np.floor(df.lat / CELL).astype(int)
    gj = np.floor(df.lon / CELL).astype(int)
    return df.assign(gi=gi, gj=gj).groupby(["gi","gj"]).w.sum()

base = cell_scores(road)
# Bootstrap sobre trayectos
trs = road.trayecto.unique()
B = 200
rng = np.random.default_rng(42)
boot = []
for _ in range(B):
    pick = rng.choice(trs, size=len(trs), replace=True)
    sub = pd.concat([road[road.trayecto == t] for t in pick])
    boot.append(cell_scores(sub))
boot = pd.concat(boot, axis=1).fillna(0)
res = pd.DataFrame({"score": base, "ci_lo": boot.quantile(.025, axis=1),
                    "ci_hi": boot.quantile(.975, axis=1),
                    "presence": (boot > 0).mean(axis=1)}).fillna(0)
res["score100"] = 100 * res.score / res.score.max()
res = res.reset_index()
res["lat"] = (res.gi + .5) * CELL; res["lon"] = (res.gj + .5) * CELL
# Estable = la celda aparece en >=80% de los remuestreos (robusta al trayecto concreto)
res["stable"] = res.presence >= 0.80
print(f"{len(res)} celdas con evento vial | celdas estables: {int(res.stable.sum())}")
display(res.sort_values("score100", ascending=False).head(10)
           [["lat","lon","score100","ci_lo","ci_hi","stable"]].round(2))

## C2 — Mapa coroplético por celda (figura de memoria)
Celdas coloreadas por danger score; borde marcado en celdas **estables** (robustas al bootstrap). Lectura limpia como mapa de riesgo, sin dependencia de osmnx.

In [ ]:
import branca.colormap as cm
center = [geo.lat.mean(), geo.lon.mean()]
m2 = folium.Map(location=center, zoom_start=13, tiles="cartodbpositron")
cmap = cm.LinearColormap(["#2c7bb6","#ffffbf","#d7191c"], vmin=0, vmax=100,
                         caption="Danger score acústico (0–100)")
for _, r in res[res.score100 > 0].iterrows():
    folium.Rectangle(
        bounds=[[r.gi*CELL, r.gj*CELL], [(r.gi+1)*CELL, (r.gj+1)*CELL]],
        color="#000000" if r.stable else None, weight=1.2 if r.stable else 0,
        fill=True, fill_color=cmap(r.score100), fill_opacity=0.75,
        popup=f"score={r.score100:.0f} IC[{r.ci_lo:.1f},{r.ci_hi:.1f}]",
    ).add_to(m2)
cmap.add_to(m2)
m2.save("../outputs/map_danger_choropleth.html")
print("guardado outputs/map_danger_choropleth.html")
m2

## C3 — Mapa interactivo (anexo/demo)
Capas conmutables: trayectos, puntos por clase (tamaño=confianza), heatmap ponderado, toggle de fuente. Popups con clase/confianza/hora.

In [ ]:
m1 = folium.Map(location=center, zoom_start=13, tiles="cartodbpositron")
# Trayectos como líneas tenues
tracks = gu.load_tracks(); tracks = tracks[~tracks.trayecto.isin(gu.BAD_TRAYECTOS)]
fg_routes = folium.FeatureGroup(name="Trayectos", show=True)
for tr, d in tracks.sort_values("time").groupby("trayecto"):
    folium.PolyLine(d[["lat","lon"]].values.tolist(), color="#888", weight=2, opacity=.4).add_to(fg_routes)
fg_routes.add_to(m1)
# Heatmap ponderado Horn+Siren
fg_heat = folium.FeatureGroup(name="Heatmap peligro", show=True)
HeatMap(road[["lat","lon","w"]].values.tolist(), radius=18, blur=22).add_to(fg_heat)
fg_heat.add_to(m1)
# Puntos por clase vial
colors = {0: "#e67e22", 1: "#c0392b"}
for c in gu.ROAD_CLASSES:
    fg = folium.FeatureGroup(name=f"{gu.CLASS_NAMES[c]}", show=(c==1))
    for _, r in road[road["class"]==c].iterrows():
        folium.CircleMarker(
            [r.lat, r.lon], radius=3 + 7*r.confidence, color=colors[c],
            fill=True, fill_opacity=.6,
            popup=f"{gu.CLASS_NAMES[c]} | conf={r.confidence:.2f} | {r.source} | {r.t_start:%H:%M:%S}",
        ).add_to(fg)
    fg.add_to(m1)
folium.LayerControl(collapsed=False).add_to(m1)
m1.save("../outputs/map_interactive.html")
print("guardado outputs/map_interactive.html")
m1

## C4 — Trayectoria coloreada por velocidad + eventos viales
Liga visualmente congestión (baja velocidad) y bocinazos. Ejemplo en un trayecto.

In [ ]:
from branca.colormap import LinearColormap
tracks_sp = gu.add_speed(tracks)
tracks_sp.loc[tracks_sp.speed_kmh > 120, "speed_kmh"] = np.nan
example = tracks_sp.trayecto.value_counts().index[0]
seg = tracks_sp[tracks_sp.trayecto == example].sort_values("time")
m4 = folium.Map(location=[seg.lat.mean(), seg.lon.mean()], zoom_start=14, tiles="cartodbpositron")
vmap = LinearColormap(["#d7191c","#ffffbf","#1a9641"], vmin=0, vmax=80, caption="velocidad km/h")
pts = seg[["lat","lon"]].values
for i in range(len(pts)-1):
    v = seg.speed_kmh.iloc[i+1]
    folium.PolyLine(pts[i:i+2].tolist(), color=vmap(0 if np.isnan(v) else min(v,80)),
                    weight=5, opacity=.9).add_to(m4)
for _, r in road[road.trayecto == example].iterrows():
    folium.CircleMarker([r.lat, r.lon], radius=6, color="black", fill=True,
                        fill_color=colors.get(r["class"],"#000"), fill_opacity=1,
                        popup=f"{gu.CLASS_NAMES[r['class']]} conf={r.confidence:.2f}").add_to(m4)
vmap.add_to(m4)
m4.save("../outputs/map_speed_trajectory.html")
print(f"trayecto={example} -> outputs/map_speed_trajectory.html")
m4

### Salidas del notebook
- `outputs/map_danger_choropleth.html` — **figura estrella** (riesgo por celda + estabilidad).
- `outputs/map_interactive.html` — demo navegable con capas.
- `outputs/map_speed_trajectory.html` — velocidad vs eventos.
- `res` (score por celda + IC). **Limitaciones:** Siren muy escasa; precisión=1.0 si no hay validación; incertidumbre espacial del join no propagada al borde de celda.